In [ ]:
from pathlib import Path

# healthy

maps_dir = Path("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/hfischer/WALINET_Vol5_Noisy_after_Walinet_Unet_abs_max/maps/Orig")

# MS 180

#maps_dir = Path("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/hfischer/ProtonFits/7T/WALINET_7T_MS_180_Unet_1.1_558_6Layer_3/maps/Orig")


In [ ]:
from pathlib import Path

mean_std_path = Path("../MetabModes/Metab_Mean_STD.txt")

sim_params = {}

with open(mean_std_path, "r") as f:
    for line in f:
        name, mean, std = line.strip().split(",")

        sim_params[name.strip()] = {
            "mean": float(mean),
            "std": float(std),
        }

print(sim_params)

In [ ]:
from pathlib import Path

available = {
    p.name.replace("_amp_map.nii.gz", "")
    for p in maps_dir.glob("*_amp_map.nii.gz")
}

manual_alias = {
    "Scy": "Scyllo",
    "THG": "TwoHG",
}

invivo_alias = {}

for sim_name in sim_params:
    if sim_name in available:
        invivo_alias[sim_name] = sim_name
    elif sim_name in manual_alias and manual_alias[sim_name] in available:
        invivo_alias[sim_name] = manual_alias[sim_name]
    else:
        invivo_alias[sim_name] = None

for k, v in invivo_alias.items():
    print(f"{k:6s} -> {v}")

In [ ]:
from pathlib import Path
import nibabel as nib
import numpy as np

crlb_threshold = 10000  # z.B. 20, 30, 50

invivo_values = {}

for sim_name, params in sim_params.items():
    invivo_name = invivo_alias.get(sim_name)

    if invivo_name is None:
        print(f"Skipping {sim_name}: no in-vivo map")
        continue

    amp_path = maps_dir / f"{invivo_name}_amp_map.nii.gz"
    sd_path  = maps_dir / f"{invivo_name}_sd_map.nii.gz"

    if not amp_path.exists() or not sd_path.exists():
        print(f"Missing files for {sim_name} -> {invivo_name}")
        continue

    amp = nib.load(amp_path).get_fdata()
    sd = nib.load(sd_path).get_fdata()

    valid = (
        np.isfinite(amp)
        & np.isfinite(sd)
        & (amp > 0)
        #& (sd > 0)
        #& (sd < crlb_threshold)
    )

    invivo_values[sim_name] = amp[valid]

    print(
        f"{sim_name:6s} -> {invivo_name:8s}: "
        f"{invivo_values[sim_name].size} voxels"
    )

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n_sim = 100_000

common_names = [name for name in sim_params if invivo_alias.get(name) is not None]

# -------------------------
# Simulation: pro Spektrum normieren
# -------------------------
sim_matrix = []

for name in common_names:
    mu = sim_params[name]["mean"]
    sd = sim_params[name]["std"]
    vals = rng.normal(mu, sd, size=n_sim)
    vals = np.clip(vals, 0, None)
    sim_matrix.append(vals)

sim_matrix = np.stack(sim_matrix, axis=1)
sim_sum = sim_matrix.sum(axis=1, keepdims=True)

valid_sim = sim_sum[:, 0] > 0
sim_norm = sim_matrix[valid_sim] / sim_sum[valid_sim]

# -------------------------
# In vivo: Maps laden, dann pro Voxel normieren
# -------------------------
amp_maps = {}

for name in common_names:
    invivo_name = invivo_alias[name]
    amp_path = maps_dir / f"{invivo_name}_amp_map.nii.gz"

    if not amp_path.exists():
        print(f"Missing {name} -> {invivo_name}, skipping")
        continue

    amp_maps[name] = nib.load(amp_path).get_fdata()

# Nur Namen behalten, die wirklich geladen wurden
common_names = [name for name in common_names if name in amp_maps]

# Gemeinsame valide Voxels
valid = np.ones(next(iter(amp_maps.values())).shape, dtype=bool)

for name in common_names:
    amp = amp_maps[name]
    valid &= np.isfinite(amp) & (amp > 0)

invivo_matrix = np.stack(
    [amp_maps[name][valid] for name in common_names],
    axis=1
)

invivo_sum = invivo_matrix.sum(axis=1, keepdims=True)
valid_invivo = invivo_sum[:, 0] > 0

invivo_norm = invivo_matrix[valid_invivo] / invivo_sum[valid_invivo]

print("Valid in-vivo voxels:", invivo_norm.shape[0])

# -------------------------
# Plot
# -------------------------
positions = np.arange(len(common_names))
width = 0.35

fig, ax = plt.subplots(figsize=(15, 5))

bp_sim = ax.boxplot(
    [sim_norm[:, i] for i in range(len(common_names))],
    positions=positions - width / 2,
    widths=0.28,
    showfliers=False,
    patch_artist=True,
)

bp_inv = ax.boxplot(
    [invivo_norm[:, i] for i in range(len(common_names))],
    positions=positions + width / 2,
    widths=0.28,
    showfliers=False,
    patch_artist=True,
)

for box in bp_sim["boxes"]:
    box.set(facecolor="tab:blue", alpha=0.35)

for box in bp_inv["boxes"]:
    box.set(facecolor="tab:orange", alpha=0.35)

ax.set_xticks(positions)
ax.set_xticklabels(common_names, rotation=45, ha="right")
ax.set_ylabel("Relative amplitude per voxel")
ax.set_title("Simulation vs in-vivo relative metabolite distributions")

ax.legend(
    handles=[
        plt.Line2D([0], [0], color="tab:blue", lw=8, alpha=0.35, label="simulation"),
        plt.Line2D([0], [0], color="tab:orange", lw=8, alpha=0.35, label="in vivo"),
    ],
    loc="upper right",
)

fig.tight_layout()
plt.show()